In [16]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA, FastICA
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_validate

import torch
import torch.nn as nn

from scipy.stats import skew, kurtosis

import seaborn as sns
import pandas as pd
import matplotlib.colors as mcolors

from tqdm import tqdm
from os import listdir
from os.path import join
import os

# Detection using two subbands

In [ ]:
cover_path = f"dataset/cover/ica_all_pairs/{10}_{8}"
stego_path = f"dataset/hinet/ica_all_pairs/{10}_{8}"

loc = sorted(listdir(cover_path))
los = sorted(listdir(stego_path))

print(len(loc), len(los))

samples = np.zeros((len(loc), 2, 2, 4)) # We assume len(loc) = len(los)
for i in tqdm(range(len(loc))):
    tmp = np.load(join(cover_path, loc[i]))[:,:]
    samples[i,0,:,0] = tmp.mean(axis=0)
    samples[i,0,:,1] = tmp.std(axis=0)
    samples[i,0,:,2] = skew(tmp, axis=0)
    samples[i,0,:,3] = kurtosis(tmp, axis=0)

for i in tqdm(range(len(los))):
    tmp = np.load(join(stego_path, los[i]))[:,:]
    samples[i,1,:,0] = tmp.mean(axis=0)
    samples[i,1,:,1] = tmp.std(axis=0)
    samples[i,1,:,2] = skew(tmp, axis=0)
    samples[i,1,:,3] = kurtosis(tmp, axis=0)

samples_training = np.concatenate([samples[:,0], samples[:,1]])
samples_training = samples_training.reshape(-1,samples_training.shape[1]*samples_training.shape[2])
labels = np.concatenate([np.zeros((len(loc))), np.ones((len(los)))])

In [ ]:
lr = LogisticRegression(max_iter=5_000)
cv_results = cross_validate(lr, samples_training,labels, cv=5)

print(cv_results["test_score"], cv_results["test_score"].mean(), cv_results["test_score"].std())

# Detection with the SRM features

In [ ]:
cover_path = "dataset/cover/spam"
stego_path = "dataset/hinet/spam"

loc = sorted(listdir(cover_path))
los = sorted(listdir(stego_path))

print(len(loc))
print(len(los))

samples = np.zeros((len(loc), 2, 5746))
for i in tqdm(range(len(loc))):
    samples[i,0] = np.load(join(cover_path, loc[i]))
for i in tqdm(range(len(los))):
    samples[i,1] = np.load(join(stego_path, los[i]))
samples_training = np.concatenate([samples[:,0], samples[:,1]])
labels = np.concatenate([np.zeros((len(loc))), np.ones((len(los)))])

In [ ]:
lr = LogisticRegression(max_iter=2000)
cv_results = cross_validate(lr, samples_training, labels, cv=5)

print(cv_results["test_score"], cv_results["test_score"].mean(), cv_results["test_score"].std())